# Create Refresh-Safe Agent Workbook

## Purpose

This notebook creates the operational workbook used by agents to review missed-refund cases.

The workbook combines essential source case information with persistent agent updates while ensuring that source refreshes do not overwrite agent-entered notes, assignments, statuses or outcomes.

In [1]:
import pandas as pd
from pathlib import Path

In [2]:
operational_folder = Path("../data/operational")

source_path = operational_folder / "source_cases.csv"
agent_updates_path = operational_folder / "agent_updates.csv"
agent_workbook_path = operational_folder / "agent_case_workbook.xlsx"

print(f"Source file found: {source_path.exists()}")
print(f"Agent updates file found: {agent_updates_path.exists()}")
print(f"Workbook already exists: {agent_workbook_path.exists()}")

Source file found: True
Agent updates file found: True
Workbook already exists: True


In [3]:
source_df = pd.read_csv(
    source_path,
    dtype={
        "Case ID": "string",
        "Policy Number": "string",
        "Client Number": "string",
    },
)

agent_updates_df = pd.read_csv(
    agent_updates_path,
    dtype={"Case ID": "string"},
)


print(f"Source rows: {len(source_df)}")
print(f"Agent update rows: {len(agent_updates_df)}")

Source rows: 2871
Agent update rows: 2871


In [4]:
source_duplicates = source_df["Case ID"].duplicated().sum()
update_duplicates = agent_updates_df["Case ID"].duplicated().sum()

missing_update_records = (
    ~source_df["Case ID"].isin(agent_updates_df["Case ID"])
).sum()

unexpected_update_records = (
    ~agent_updates_df["Case ID"].isin(source_df["Case ID"])
).sum()

print(f"Duplicate source Case IDs: {source_duplicates}")
print(f"Duplicate agent Case IDs: {update_duplicates}")
print(f"Source cases without agent records: {missing_update_records}")
print(f"Agent records without source cases: {unexpected_update_records}")

Duplicate source Case IDs: 0
Duplicate agent Case IDs: 0
Source cases without agent records: 0
Agent records without source cases: 0


In [5]:
manual_update_columns = [
    "Case ID",
    "Agent Working",
    "Agent Notes",
    "Case Status",
    "Final Outcome",
]

if agent_workbook_path.exists():
    preserved_updates_df = pd.read_excel(
        agent_workbook_path,
        sheet_name="Case Review",
        usecols=manual_update_columns,
        dtype={"Case ID": "string"},
    )

    print("Loaded agent updates from the existing workbook.")
else:
    preserved_updates_df = agent_updates_df[
        manual_update_columns
    ].copy()

    print("Loaded initial updates from agent_updates.csv.")

Loaded agent updates from the existing workbook.


In [6]:
agent_source_columns = [
    "Case ID",
    "Policy Number",
    "Client Number",
    "Outstanding Amount",
    "Cancellation Status",
]

agent_update_columns = [
    "Case ID",
    "Agent Working",
    "Agent Notes",
    "Case Status",
    "Final Outcome",
]

agent_workbook_df = source_df[
    agent_source_columns
].merge(
    preserved_updates_df[agent_update_columns],
    on="Case ID",
    how="left",
    validate="one_to_one",
)

print(f"Agent workbook rows: {len(agent_workbook_df)}")
print(f"Agent workbook columns: {len(agent_workbook_df.columns)}")

Agent workbook rows: 2871
Agent workbook columns: 9


In [7]:
editable_fields = [
    "Agent Working",
    "Agent Notes",
    "Case Status",
    "Final Outcome",
]

backend_updates = agent_updates_df.set_index(
    "Case ID"
).copy()

workbook_updates = preserved_updates_df.set_index(
    "Case ID"
).reindex(
    backend_updates.index
)

changed_records = pd.Series(
    False,
    index=backend_updates.index,
)

for field in editable_fields:
    old_values = (
        backend_updates[field]
        .fillna("")
        .astype(str)
    )

    new_values = (
        workbook_updates[field]
        .fillna("")
        .astype(str)
    )

    changed_records = (
        changed_records
        | old_values.ne(new_values)
    )

    backend_updates[field] = workbook_updates[field]

print(
    f"Agent records changed: "
    f"{changed_records.sum()}"
)

Agent records changed: 0


In [8]:
synchronisation_date = (
    pd.Timestamp.today()
    .strftime("%Y-%m-%d")
)

backend_updates.loc[
    changed_records,
    "Last Updated Date",
] = synchronisation_date

completed_records = (
    backend_updates["Case Status"]
    == "Completed"
)

missing_completed_agent = (
    backend_updates["Agent Working"]
    .fillna("")
    .eq("")
)

missing_completed_outcome = (
    backend_updates["Final Outcome"]
    .fillna("")
    .eq("")
)

invalid_completed_records = (
    completed_records
    & (
        missing_completed_agent
        | missing_completed_outcome
    )
)

if invalid_completed_records.any():
    raise ValueError(
        "Completed cases require an agent "
        "and a final outcome."
    )

completion_date_missing = (
    backend_updates["Completion Date"].isna()
    | backend_updates["Completion Date"].eq("")
)

newly_completed_records = (
    completed_records
    & completion_date_missing
)

backend_updates.loc[
    newly_completed_records,
    "Completed By",
] = backend_updates.loc[
    newly_completed_records,
    "Agent Working",
]

backend_updates.loc[
    newly_completed_records,
    "Completion Date",
] = synchronisation_date

synced_agent_updates_df = (
    backend_updates
    .reset_index()
    [agent_updates_df.columns]
)

synced_agent_updates_df.to_csv(
    agent_updates_path,
    index=False,
)

agent_updates_df = (
    synced_agent_updates_df.copy()
)

print(
    f"Agent records synchronised: "
    f"{len(agent_updates_df)}"
)
print(
    f"Records updated: "
    f"{changed_records.sum()}"
)
print(
    f"Newly completed records: "
    f"{newly_completed_records.sum()}"
)
print(
    f"Synchronisation date: "
    f"{synchronisation_date}"
)

Agent records synchronised: 2871
Records updated: 0
Newly completed records: 0
Synchronisation date: 2026-07-21


In [9]:
backend_check_df = pd.read_csv(
    agent_updates_path,
    dtype={"Case ID": "string"},
)

backend_check_df.loc[
    backend_check_df["Case ID"] == "DH000050",
    [
        "Case ID",
        "Agent Working",
        "Agent Notes",
        "Case Status",
        "Last Updated Date",
    ],
]

,Case ID,Agent Working,Agent Notes,Case Status,Last Updated Date
49,DH000050,DH001,Senior review completed; refund processed.,Completed,2026-07-21


In [10]:
system_maintained_columns = [
    "Completed By",
    "Completion Date",
    "Last Updated Date",
]

for column in system_maintained_columns:
    backend_updates[column] = (
        backend_updates[column]
        .astype("object")
    )

In [11]:
agent_workbook_df.loc[
    agent_workbook_df["Case ID"] == "DH000050"
]

,Case ID,Policy Number,Client Number,Outstanding Amount,Cancellation Status,Agent Working,Agent Notes,Case Status,Final Outcome
49,DH000050,PET000050,CL000051,7.06,Void,DH001,Senior review completed; refund processed.,Completed,Refund Processed


In [12]:
with pd.ExcelWriter(
    agent_workbook_path,
    engine="openpyxl",
) as writer:
    agent_workbook_df.to_excel(
        writer,
        sheet_name="Case Review",
        index=False,
    )

print(f"Created agent workbook: {agent_workbook_path}")

Created agent workbook: ..\data\operational\agent_case_workbook.xlsx


In [13]:
preservation_test_df = pd.read_excel(
    agent_workbook_path,
    sheet_name="Case Review",
    dtype={"Case ID": "string"},
)

preservation_test_df.loc[
    preservation_test_df["Case ID"] == "DH000050",
    [
        "Case ID",
        "Agent Working",
        "Agent Notes",
        "Case Status",
    ],
]

,Case ID,Agent Working,Agent Notes,Case Status
49,DH000050,DH001,Senior review completed; refund processed.,Completed


In [14]:
agents_reference_path = (
    Path("../data/reference/debt_held_agents.csv")
)

agents_reference_df = pd.read_csv(
    agents_reference_path
)

agents_reference_df

,Agent ID,Agent Role,Active
0,DH001,Debt Held Agent,True
1,DH002,Debt Held Agent,True


In [15]:
active_agent_ids = agents_reference_df.loc[
    agents_reference_df["Active"] == True,
    "Agent ID",
].tolist()

case_status_options = [
    "Open",
    "Awaiting Senior Review",
    "Awaiting Other Department",
    "Completed",
]

final_outcome_options = [
    "Refund Processed",
    "Refund Already Processed",
    "No Refund Due",
]

print(f"Active agents: {active_agent_ids}")
print(f"Case statuses: {case_status_options}")
print(f"Final outcomes: {final_outcome_options}")

Active agents: ['DH001', 'DH002']
Case statuses: ['Open', 'Awaiting Senior Review', 'Awaiting Other Department', 'Completed']
Final outcomes: ['Refund Processed', 'Refund Already Processed', 'No Refund Due']


In [16]:
workbook_check_df = pd.read_excel(
    agent_workbook_path,
    sheet_name="Case Review",
    dtype={
        "Case ID": "string",
        "Policy Number": "string",
        "Client Number": "string",
    },
)

print(f"Workbook rows: {len(workbook_check_df)}")
print(f"Workbook columns: {len(workbook_check_df.columns)}")
print(
    "Duplicate Case IDs:",
    workbook_check_df["Case ID"].duplicated().sum(),
)

Workbook rows: 2871
Workbook columns: 9
Duplicate Case IDs: 0


## Workbook Formatting and Validation

The following steps format the agent-facing workbook and add controlled dropdown lists to support accurate and consistent case updates.

In [17]:
from openpyxl import load_workbook
from openpyxl.styles import Alignment, Font, PatternFill
from openpyxl.worksheet.datavalidation import DataValidation
from openpyxl.worksheet.table import Table, TableStyleInfo

workbook = load_workbook(agent_workbook_path)
case_review_sheet = workbook["Case Review"]

print(
    f"Formatting {case_review_sheet.max_row - 1} case rows."
)

Formatting 2871 case rows.


In [18]:
case_review_sheet.freeze_panes = "A2"
case_review_sheet.sheet_view.showGridLines = False

column_widths = {
    "A": 14,
    "B": 16,
    "C": 15,
    "D": 22,
    "E": 22,
    "F": 16,
    "G": 55,
    "H": 28,
    "I": 26,
}

for column_letter, width in column_widths.items():
    case_review_sheet.column_dimensions[
        column_letter
    ].width = width

table_reference = (
    f"A1:I{case_review_sheet.max_row}"
)

if "CaseReviewTable" not in case_review_sheet.tables:
    case_table = Table(
        displayName="CaseReviewTable",
        ref=table_reference,
    )

    case_table.tableStyleInfo = TableStyleInfo(
        name="TableStyleMedium2",
        showFirstColumn=False,
        showLastColumn=False,
        showRowStripes=True,
        showColumnStripes=False,
    )

    case_review_sheet.add_table(case_table)

workbook.save(agent_workbook_path)

print("Applied workbook layout and table formatting.")

Applied workbook layout and table formatting.


In [19]:
editable_fill = PatternFill(
    fill_type="solid",
    fgColor="FFF2CC",
)

for row_number in range(
    2,
    case_review_sheet.max_row + 1,
):
    case_review_sheet.cell(
        row=row_number,
        column=4,
    ).number_format = '£#,##0.00'

    for column_number in range(6, 10):
        case_review_sheet.cell(
            row=row_number,
            column=column_number,
        ).fill = editable_fill

    case_review_sheet.cell(
        row=row_number,
        column=7,
    ).alignment = Alignment(
        wrap_text=True,
        vertical="top",
    )

workbook.save(agent_workbook_path)

print("Formatted currency and editable agent fields.")

Formatted currency and editable agent fields.


In [20]:
agent_validation = DataValidation(
    type="list",
    formula1=(
        f'"{",".join(active_agent_ids)}"'
    ),
    allow_blank=True,
)

status_validation = DataValidation(
    type="list",
    formula1=(
        f'"{",".join(case_status_options)}"'
    ),
    allow_blank=False,
)

outcome_validation = DataValidation(
    type="list",
    formula1=(
        f'"{",".join(final_outcome_options)}"'
    ),
    allow_blank=True,
)

for validation in [
    agent_validation,
    status_validation,
    outcome_validation,
]:
    validation.errorTitle = "Invalid selection"
    validation.error = (
        "Please select a value from the dropdown list."
    )
    validation.showErrorMessage = True

case_review_sheet.add_data_validation(
    agent_validation
)
case_review_sheet.add_data_validation(
    status_validation
)
case_review_sheet.add_data_validation(
    outcome_validation
)

agent_validation.add(
    f"F2:F{case_review_sheet.max_row}"
)

status_validation.add(
    f"H2:H{case_review_sheet.max_row}"
)

outcome_validation.add(
    f"I2:I{case_review_sheet.max_row}"
)

workbook.save(agent_workbook_path)

print("Added agent, status and outcome dropdowns.")

Added agent, status and outcome dropdowns.


In [21]:
if "Instructions" in workbook.sheetnames:
    del workbook["Instructions"]

instructions_sheet = workbook.create_sheet(
    "Instructions",
    0,
)

instructions_sheet["A1"] = (
    "Missed Refund Agent Case Workbook"
)
instructions_sheet["A2"] = (
    "Synthetic portfolio data — no real customer "
    "information is included."
)

instruction_rows = [
    (
        "Editable fields",
        "Only the pale-yellow columns should be updated.",
    ),
    (
        "Assignment",
        "Select your Agent ID when reviewing a case. "
        "Remove it if you cannot complete the case.",
    ),
    (
        "Agent notes",
        "Record a short operational note only when needed.",
    ),
    (
        "Case status",
        "Use Open, Awaiting Senior Review, "
        "Awaiting Other Department or Completed.",
    ),
    (
        "Final outcome",
        "Select an outcome only when the case is completed.",
    ),
    (
        "Refresh",
        "Save and close this workbook before the refresh "
        "notebook is run.",
    ),
]

for row_number, values in enumerate(
    instruction_rows,
    start=4,
):
    instructions_sheet.cell(
        row=row_number,
        column=1,
        value=values[0],
    )
    instructions_sheet.cell(
        row=row_number,
        column=2,
        value=values[1],
    )

instructions_sheet["A1"].font = Font(
    bold=True,
    color="FFFFFF",
    size=16,
)
instructions_sheet["A1"].fill = PatternFill(
    fill_type="solid",
    fgColor="1F4E78",
)
instructions_sheet.merge_cells("A1:B1")

instructions_sheet["A2"].font = Font(
    italic=True,
    color="666666",
)
instructions_sheet.merge_cells("A2:B2")

for row_number in range(4, 10):
    instructions_sheet.cell(
        row=row_number,
        column=1,
    ).font = Font(bold=True)

    instructions_sheet.cell(
        row=row_number,
        column=2,
    ).alignment = Alignment(
        wrap_text=True,
        vertical="top",
    )

instructions_sheet.column_dimensions["A"].width = 22
instructions_sheet.column_dimensions["B"].width = 85
instructions_sheet.sheet_view.showGridLines = False

workbook.save(agent_workbook_path)

print("Added workbook instructions sheet.")

Added workbook instructions sheet.


In [22]:
final_workbook = load_workbook(
    agent_workbook_path,
    data_only=False,
)

final_case_sheet = final_workbook["Case Review"]

print(f"Workbook sheets: {final_workbook.sheetnames}")
print(
    f"Case rows: {final_case_sheet.max_row - 1}"
)
print(
    f"Case columns: {final_case_sheet.max_column}"
)
print(
    f"Excel tables: {list(final_case_sheet.tables.keys())}"
)
print(
    "Dropdown validations:",
    len(
        final_case_sheet
        .data_validations
        .dataValidation
    ),
)

Workbook sheets: ['Instructions', 'Case Review']
Case rows: 2871
Case columns: 9
Excel tables: ['CaseReviewTable']
Dropdown validations: 3


In [23]:
current_workbook_check_df = pd.read_excel(
    agent_workbook_path,
    sheet_name="Case Review",
    dtype={"Case ID": "string"},
)

current_workbook_check_df.loc[
    current_workbook_check_df["Case ID"] == "DH000050",
    [
        "Case ID",
        "Agent Working",
        "Agent Notes",
        "Case Status",
        "Final Outcome",
    ],
]

,Case ID,Agent Working,Agent Notes,Case Status,Final Outcome
49,DH000050,DH001,Senior review completed; refund processed.,Completed,Refund Processed


In [24]:
latest_backend_check_df = pd.read_csv(
    agent_updates_path,
    dtype={"Case ID": "string"},
)

latest_backend_check_df.loc[
    latest_backend_check_df["Case ID"] == "DH000050",
    [
        "Case ID",
        "Agent Working",
        "Agent Notes",
        "Case Status",
        "Final Outcome",
        "Completed By",
        "Completion Date",
        "Last Updated Date",
    ],
]

,Case ID,Agent Working,Agent Notes,Case Status,Final Outcome,Completed By,Completion Date,Last Updated Date
49,DH000050,DH001,Senior review completed; refund processed.,Completed,Refund Processed,DH001,2026-07-21,2026-07-21


In [25]:
completed_case_check = pd.read_csv(
    agent_updates_path,
    dtype={"Case ID": "string"},
).loc[
    lambda df: df["Case ID"] == "DH000050"
].iloc[0]

fields_to_check = [
    "Case Status",
    "Final Outcome",
    "Completed By",
    "Completion Date",
    "Last Updated Date",
]

for field in fields_to_check:
    print(
        f"{field}: "
        f"{completed_case_check[field]}"
    )

Case Status: Completed
Final Outcome: Refund Processed
Completed By: DH001
Completion Date: 2026-07-21
Last Updated Date: 2026-07-21
